# 18. Reranking with a Cross-Encoder

Retrieve broadly, then let a cross-encoder pick the best few.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Embedding search compares two **independently computed** vectors - fast but coarse. A **cross-encoder reranker** reads the query and a passage *together* and outputs a precise relevance score. Because that is slower, we use a two-stage funnel: a fast retriever fetches ~15-50 candidates, then the reranker re-orders them and we keep the best 3-5. Typically the biggest precision gain for a modest latency cost.

## How it works

1. Retrieve a wide candidate set (k=15) with the fast bi-encoder
2. Score each (query, passage) pair with a cross-encoder
3. Sort by reranker score
4. Keep the top few for the LLM
5. Optionally rerank with the LLM itself

```text
 query --> bi-encoder --> 15 candidates --> cross-encoder(query, passage) --> re-sorted --> top 3 --> LLM
           (fast, coarse)                    (slow, precise)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 12:36:42.937000 14216 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, hits, top_n=3):
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(scores, hits), key=lambda p: -p[0])[:top_n]
    return [{**h, "score": float(s)} for s, h in ranked]

query = "Why are cross-encoders only applied to a few candidates?"
candidates = store.search(query, 15)
print("--- before rerank (bi-encoder)"); show(candidates[:3], width=100)
final = rerank(query, candidates)
print("--- after rerank (cross-encoder)"); show(final, width=100)

--- before rerank (bi-encoder)
0.564  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. ...
0.362  is usually measured with cosine similarity, which compares direction rather than length. If the vect...
0.290  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted s...


--- after rerank (cross-encoder)
6.976  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. ...
-3.484  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted s...
-10.847  them, because the model can ignore the context or over-generalise from it. Grounding techniques incl...


In [3]:
print(answer(query, [h["text"] for h in final]))

According to the provided context, cross-encoders are slow. Therefore, they are applied only to the top 20 to 50 candidates, which is likely because this number provides a good balance between accuracy and computational efficiency.


### LLM reranking (no extra model)
Ask the LLM to score each candidate 0-10. Slower, but works with any model.

In [4]:
def llm_rerank(query, hits, top_n=3):
    scored = []
    for h in hits:
        out = llm(f"Rate from 0 to 10 how well the passage answers the question. Reply with the number only.\n\nQuestion: {query}\n\nPassage: {h['text']}")
        m = re.search(r"\d+", out)
        scored.append((int(m.group()) if m else 0, h))
    return [h for _, h in sorted(scored, key=lambda p: -p[0])[:top_n]]

show(llm_rerank(query, candidates[:6]), width=100)

0.290  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted s...
0.564  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. ...
0.362  is usually measured with cosine similarity, which compares direction rather than length. If the vect...


## Evaluate: did the evidence move up?

In [5]:
def reranked_search(q, k=3, pool=15):
    return rerank(q, store.search(q, pool), top_n=k)

compare_retrievers({"bi-encoder only": store.search, "bi-encoder + cross-encoder": reranked_search}, k=3)

,hit@1,hit@3,MRR
method,,,
bi-encoder only,0.80,1.0,0.89
bi-encoder + cross-encoder,0.95,1.0,0.97


## Effect of the candidate pool size
A bigger pool gives the reranker more to work with, at a latency cost.

In [6]:
import pandas as pd, time
rows = []
for pool in [3, 5, 10, 15, len(chunks)]:
    t = time.time()
    r = evaluate_retriever(lambda q, k, pool=pool: rerank(q, store.search(q, pool), top_n=k), k=3)
    rows.append({"pool": pool, "hit@3": r["hit_rate@3"], "MRR": r["MRR"], "seconds": round(time.time() - t, 1)})
pd.DataFrame(rows).set_index("pool")

,hit@3,MRR,seconds
pool,,,
3,1.0,0.97,0.6
5,1.0,0.97,0.6
10,1.0,0.97,0.9
15,1.0,0.97,1.2
19,1.0,0.97,1.5


## See the reordering for one query

In [7]:
q = "What is mean reciprocal rank?"
cands = store.search(q, 8)
rr = rerank(q, cands, top_n=8)
for before, h in enumerate(cands, 1):
    after = next(i for i, x in enumerate(rr, 1) if x["idx"] == h["idx"])
    print(f"bi-encoder rank {before} -> reranked {after}   {h['text'][:70]!r}")

bi-encoder rank 1 -> reranked 2   't paraphrases and meaning. Hybrid search runs both and merges the rank'
bi-encoder rank 2 -> reranked 5   'r and outputs a relevance score, which is far more accurate than compa'
bi-encoder rank 3 -> reranked 1   'earches with the embedding of that answer, because an answer looks mor'
bi-encoder rank 4 -> reranked 4   'is usually measured with cosine similarity, which compares direction r'
bi-encoder rank 5 -> reranked 3   'the embedding. A chunk size of around 500 to 1000 characters with 10 t'
bi-encoder rank 6 -> reranked 6   'has four stages: ingest documents, split them into chunks, index the c'
bi-encoder rank 7 -> reranked 7   'index, optionally reranking and finally generating the answer, with ge'
bi-encoder rank 8 -> reranked 8   'ten vague, short or badly phrased, so the query is rewritten before re'


## Latency of reranking

In [8]:
import time
t = time.time(); store.search(q, 15); a = time.time() - t
t = time.time(); rerank(q, store.search(q, 15)); b = time.time() - t
print(f"retrieve only: {a * 1000:.0f} ms   retrieve + rerank 15: {b * 1000:.0f} ms")

retrieve only: 29 ms   retrieve + rerank 15: 83 ms


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Large precision boost | Adds latency proportional to pool size |
| Small open models work well | Max input length truncates long passages |
| Drop-in after any retriever | Needs the extra model |

## When to use it

Always consider it - it is the most cost-effective way to make the top 3 results good. Part of the must-do core.

## Experiments to try

- Combine with fusion retrieval (17): fuse, then rerank.
- Try `BAAI/bge-reranker-base`.
- Rerank at pool sizes 5/20/50 and plot the latency/quality curve.

## Key takeaways

- Two-stage retrieval: recall first, precision second.
- Cross-encoders see query and passage together.
- Tune the pool size.

---
**Next:** [19. Multi-faceted Filtering](../19_multi_faceted_filtering/multi_faceted_filtering.ipynb)